In [2]:
import psycopg2
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import silhouette_score, silhouette_samples

In [6]:
# 1. Load and Merge
df = pd.read_csv('data/family_atlas.csv')
names_df = pd.read_csv('data/component.summary.tsv', sep='\t')
df = df.merge(names_df[['component', 'domain']], on='component', how='left')

# 2. Calculate Silhouette Scores
# We only pass the coordinates to the algorithm, but the output order 
# matches the input row order of 'df' exactly.
scores = silhouette_samples(df[['umap_x', 'umap_y']], df['domain'])

# 3. Create the results dataframe by copying the original df and adding the scores
# This preserves ALL original columns (sequence name, family, etc.)
sil_df = df.copy()
sil_df['silhouette'] = scores

# 4. Save the full metadata + scores
sil_df.to_csv('domain_silhouette_scores_with_metadata.csv', index=False)

In [8]:
outliers_df = sil_df[sil_df['silhouette'] < -0.1]
outliers_df = outliers_df[outliers_df['domain'] != 'Alpha/Beta Hydrolase']
outliers_df.to_csv('domain_outliers.csv', index=False)